In [ ]:
!pip install -q chess ipywidgets

In [ ]:
import math, random
import chess, chess.svg
import ipywidgets as w
from IPython.display import display, clear_output, SVG

VAL = {chess.PAWN: 100, chess.KNIGHT: 320, chess.BISHOP: 330,
       chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}

def evaluate(b, d):
    if b.is_checkmate():
        return -(10**5 + d) if b.turn else 10**5 + d
    if b.is_stalemate() or b.is_insufficient_material():
        return 0
    s = 0
    for sq, p in b.piece_map().items():
        f, r = chess.square_file(sq), chess.square_rank(sq)
        v = VAL[p.piece_type]
        if p.piece_type in (chess.PAWN, chess.KNIGHT):
            v += (3.5 - abs(f - 3.5)) * 4 + (3.5 - abs(r - 3.5)) * 4
        if p.piece_type == chess.PAWN:
            v += (r - 1 if p.color else 6 - r) * 5
        s += v if p.color else -v
    return s

def search(b, d, alpha, beta):
    if d == 0 or b.is_game_over():
        return evaluate(b, d)
    white = b.turn
    ms = sorted(b.legal_moves, key=lambda m: -VAL.get(b.piece_type_at(m.to_square) or 0, 0))
    best = -math.inf if white else math.inf
    for m in ms:
        b.push(m); v = search(b, d - 1, alpha, beta); b.pop()
        if white: best = max(best, v); alpha = max(alpha, v)
        else: best = min(best, v); beta = min(beta, v)
        if beta <= alpha: break
    return best

class Game:
    def __init__(s):
        s.out = w.Output(); s.msg = w.HTML()
        s.inp = w.Text(placeholder='e2e4  or  Nf3', description='Your move:')
        s.lvl = w.Dropdown(options=[('Easy', 0), ('Medium', 2), ('Hard', 3)], value=2, description='Level:')
        s.side = w.Dropdown(options=['White', 'Black'], description='You play:')
        play, new, undo = w.Button(description='Play', button_style='success'), w.Button(description='New game'), w.Button(description='Undo')
        play.on_click(lambda _: s.play()); new.on_click(lambda _: s.reset()); undo.on_click(lambda _: s.undo())
        s.inp.on_submit(lambda _: s.play())
        display(w.VBox([w.HBox([s.side, s.lvl, new, undo]), w.HBox([s.inp, play]), s.msg, s.out]))
        s.reset()

    def reset(s):
        s.b = chess.Board(); s.me = s.side.value == 'White'; s.last = None
        s.note = ''
        if not s.me: s.ai()
        s.show()

    def show(s):
        b = s.b
        if b.is_checkmate(): t = 'Checkmate - ' + ('computer wins' if b.turn == s.me else 'you win! 🎉')
        elif b.is_game_over(): t = 'Draw'
        else: t = 'Your move' + (' (check!)' if b.is_check() else '')
        s.msg.value = f'<b>{s.note} {t}</b>'
        with s.out:
            clear_output(wait=True)
            display(SVG(chess.svg.board(b, orientation=s.me, lastmove=s.last, size=420,
                                        check=b.king(b.turn) if b.is_check() else None)))

    def play(s):
        if s.b.is_game_over() or s.b.turn != s.me: return
        t = s.inp.value.strip(); m = None
        try: m = s.b.parse_san(t)
        except ValueError:
            try:
                m = chess.Move.from_uci(t.lower())
                if m not in s.b.legal_moves: m = None
            except ValueError: pass
        if m is None:
            s.msg.value = f'<b style="color:red">Illegal or unreadable move: {t!r}</b>'; return
        s.b.push(m); s.last = m; s.inp.value = ''; s.note = ''
        if not s.b.is_game_over(): s.ai()
        s.show()

    def ai(s):
        lv = s.lvl.value; ms = list(s.b.legal_moves)
        if lv == 0: pick = random.choice(ms)
        else:
            random.shuffle(ms); white, bv, pick = s.b.turn, None, ms[0]
            for m in ms:
                s.b.push(m); v = search(s.b, lv - 1, -math.inf, math.inf); s.b.pop()
                if bv is None or (v > bv if white else v < bv): bv, pick = v, m
        s.note = f'Computer played {s.b.san(pick)}.'
        s.b.push(pick); s.last = pick

    def undo(s):
        if s.b.move_stack: s.b.pop()
        if s.b.turn != s.me and s.b.move_stack: s.b.pop()
        s.last = None; s.note = ''
        if s.b.turn != s.me: s.ai()
        s.show()

Game()
